This code is my first attempt to compute effects of nearest neighbor, next nearest neighbor, and long-range interactions with arbitrary interaction energies. Since I've imported a lot of code from PHY 480, it's also generalized for 2D, although I never actaully made any 2D simulations.

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns


First codeblock is lattice class made in PHY 480, with some modifications to allow for arbitrary binding energies.

In [ ]:

class lattice:
    
    def __init__( self, num_dim, size ):
        
        self._is_init = False
        if type(num_dim) != int:
            print( "ERROR: lattice class initializaiton: num_dim must be integer.")
            return
        if len( size ) != num_dim:
            print( "ERROR: lattice class initialization: size array does not match num_dim." )
            return
        for i in range( num_dim ):
            if size[i] < 0:
                print( "ERROR: lattice class initialization: size must be positive." )
                return
        
        # initialization flag
        self._is_init = True
        # number of dimensions
        self._num_dim = int( num_dim )
        # lattice size
        self._size = np.array( size, dtype=int )
        # total number of sites (volume)
        self._volume = 1
        for i in range( num_dim ):
            self._volume *= size[i]
        self._volume = int(self._volume)
            
        # forward and backward nearest neighbor index arrays
        # (periodic boundary conditions)
        # --- Neighbor Calculation ---
        self._neighbor_forward = np.zeros((self._volume, self._num_dim), dtype=int)
        self._neighbor_backward = np.zeros((self._volume, self._num_dim), dtype=int)
        self._neighbor_nn = [[] for _ in range(self._volume)] # Using lists for flexibility

        for i in range(self._volume):
            r = self.map_index_to_vec(i)
            nn_set = set() # Use a set to automatically handle duplicates if any
            for j in range(self._num_dim):
                # Forward neighbor
                q_fwd = r.copy()
                q_fwd[j] = (q_fwd[j] + 1) % self._size[j]
                fwd_idx = self.map_vec_to_index(q_fwd)
                self._neighbor_forward[i, j] = fwd_idx
                nn_set.add(fwd_idx)

                # Backward neighbor
                q_bwd = r.copy()
                q_bwd[j] = (q_bwd[j] + self._size[j] - 1) % self._size[j]
                bwd_idx = self.map_vec_to_index(q_bwd)
                self._neighbor_backward[i, j] = bwd_idx
                nn_set.add(bwd_idx)
            self._neighbor_nn[i] = list(nn_set) # Store unique NNs

        # --- Next-Nearest Neighbor Calculation ---
        self._neighbor_nnn = [[] for _ in range(self._volume)]
        for i in range(self._volume):
            nn_of_i_set = set(self._neighbor_nn[i])
            nnn_candidates = set() # set to avoid overlap
            for j in self._neighbor_nn[i]: 
                nn_of_j_set = set(self._neighbor_nn[j]) 
                for k in nn_of_j_set:
                    if k != i and k not in nn_of_i_set:
                        nnn_candidates.add(k)
            self._neighbor_nnn[i] = list(nnn_candidates)
        
        self._is_init = True

    def map_vec_to_index( self, r ):
        
        if self._is_init == False:
            print( "ERROR: attempt to call map_vec_to_index on an unitialized lattice instance.")
            return -1
        if len(r) != self._num_dim:
            print( "ERROR: in map_vec_to_index vector has wrong dimensions:", len(r) )
            return -2
        for i in range(self._num_dim):
            if r[i]<0 or r[i]>=self._size[i]:
                print( "ERROR: in map_vec_to_index vector component", i, "is out of range." )
                return -3
            
        index = 0
        v = 1
        for i in range(self._num_dim):
            index += int(r[i])*v
            v *= self._size[i]
            
        return int(index)
    
    def map_index_to_vec( self, index ):

        if self._is_init == False:
            print( "ERROR: attempt to call map_index_to_vec on an unitialized lattice instance.")
            return -1
        if index < 0 or index >= self._volume:
            print( "ERROR: in map_index_to_vec index is out of range.")
            return -2
            
        v = int(index)
        r = np.zeros( self._num_dim, dtype=int )
        for i in range(self._num_dim):
            r[i] = int( v % self._size[i] )
            v = v // self._size[i]
        
        return r

    def make_hot_start(self, J1 = 1.0, J2 = 0.0, A = 1.0, fraction = 0.5 ):
        self._A = A
        self._J1 = J1
        self._J2 = J2
        states = (np.random.rand(self._volume) < fraction).astype(int)
        self._states = states
    
    def make_cold_start( self, J1 = 1.0, J2 = 0.0, A = 1.0, up=True ):
        self._A = A
        self._J1 = J1
        self._J2 = J2
        states = np.ones(self._volume, dtype=int)*up
        self._states = states

    
    def compute_energy( self, isite, iq=None):
    
        s_i = self._states[isite] if iq is None else iq
        
        energy_i = self._A * s_i

        interaction_nn = 0.0
        for j in self._neighbor_nn[isite]:
            s_j = self._states[j]
            interaction_nn += abs(s_i - s_j) # energy comes from difference with neighbors
        energy_i += (self._J1 / 2.0) * interaction_nn

        interaction_nnn = 0.0
        for k in self._neighbor_nnn[isite]:
            s_k = self._states[k]
            interaction_nnn += abs(s_i - s_k) # energy comes from difference with neighbors
        energy_i += (self._J2 / 2.0) * interaction_nnn

        return energy_i

    def calculate_total_energy(self):
        total_energy = 0.0
        for i in range(self._volume):
            site_energy = self.compute_energy(i)
            total_energy += site_energy
        return total_energy
    
    def calculate_delta_energy(self, isite):
        '''for Monte-Carlo methods'''
        s_i = self._states[isite]
        s_prime_i = 1 - s_i 

        energy_initial = self.compute_energy(isite, iq=s_i)
        energy_final = self.compute_energy(isite, iq=s_prime_i)

        delta_e = energy_final - energy_initial

        return delta_e

The parameter set below creates an interesting partition function using next-nearest-neighbor interactions

In [ ]:
dim = 1
N = 16
chain = lattice(dim, [N])
chain.make_cold_start(A=0.0, J1=3, J2=2)

tau = 3.5
beta = 1/tau

boltzmann_factors = []
excited_states = []
energies = []
all_chain_states = [np.array([int(bit) for bit in f'{i:0{N}b}']) for i in range(2**N)]
for state in all_chain_states:
    chain._states = state
    E = chain.calculate_total_energy()
    energies.append(E)
    boltzmann_factors.append(np.exp(-E*beta))
    excited_states.append(sum(state))

Z = sum(boltzmann_factors)

In [ ]:
chain_dict = {'state':all_chain_states, 'energy':energies, 'b_factors':boltzmann_factors, 'excited_states':excited_states}

All_States = pd.DataFrame(chain_dict)
All_States['probability'] = All_States['b_factors']/Z

#All_States['probability'].sum()
number_excited = []
state_mean_energy = []
state_max_energy = []
state_min_energy = []
for i in range(N):
    state_probability = sum(All_States.loc[All_States['excited_states'] == i]['probability'])
    state_mean_energy.append((i,All_States.loc[All_States['excited_states'] == i]['energy'].mean()))
    state_max_energy.append((i,All_States.loc[All_States['excited_states'] == i]['energy'].max()))
    state_min_energy.append((i,All_States.loc[All_States['excited_states'] == i]['energy'].min()))
    number_excited.append((i, state_probability))

prob_array = np.array(number_excited)
mean_array = np.array(state_mean_energy)
max_array = np.array(state_max_energy)
min_array = np.array(state_min_energy)

renorm_factor = prob_array[:,1].max()/max(mean_array[:,1])

plt.axvline(N/2, color='red')
plt.plot(prob_array[:,0], prob_array[:, 1], label='probability')
#plt.plot(mean_array[:,0], mean_array[:, 1]*renorm_factor, label='mean state energy (arb units)')
#plt.plot(max_array[:,0], max_array[:, 1]*renorm_factor, label='max state energy (arb units)')
#plt.plot(min_array[:,0], min_array[:, 1]*renorm_factor, label='min state energy (arb units)')
plt.xlabel('excited states')
plt.legend()
plt.grid()

Sweet! Looks like we can make some much stronger peaks simply by adding next-nearest-neighbor interactions. Of course, this plot is only a snapshot of what is really going on, and I think simulation is the only real way to see how the system evolves.

The code below is the same as above, only making it much easier for me to try out a large variety of parameter combinations. Below, I show off how temperature changes affect the probability distribution.

In [ ]:
# Production code

dim = 1
N = 16 # can't get above 20

A = 0.0
J1 = 1
J2 = 1

chain = lattice(dim, [N])
chain.make_cold_start(A=A, J1=J1, J2=J2)

meta_df = pd.DataFrame(columns=['state', 'b_factors', 'excited_states', 'probability', 'tau'])

tau_vals = np.linspace(0.85, 1.2, 30)

for tau in tau_vals:
    beta = 1/tau

    boltzmann_factors = []
    excited_states = []
    all_chain_states = [np.array([int(bit) for bit in f'{i:0{N}b}']) for i in range(2**N)]
    
    for state in all_chain_states:
        chain._states = state
        E = chain.calculate_total_energy()
        boltzmann_factors.append(np.exp(-E*beta))
        excited_states.append(sum(state))

    Z = sum(boltzmann_factors)
    chain_dict = {'state':all_chain_states, 'b_factors':boltzmann_factors, 'excited_states':excited_states}
    All_States = pd.DataFrame(chain_dict)
    All_States['probability'] = All_States['b_factors']/Z
    All_States['tau'] = tau
    meta_df = pd.concat([meta_df, All_States], ignore_index=True, sort=False)

meta_df.head()

In [ ]:
fig, ax = plt.subplots(1,1)


tau = tau_vals[0]
tau_states = meta_df[meta_df['tau'] == tau]
state_prob = []
for i in range(N):
    state_probability = sum(tau_states.loc[tau_states['excited_states'] == i]['probability'])
    state_prob.append(state_probability)

line, = plt.plot([i for i in range(N)], state_prob, label=fr'$\tau=$ {tau}')
ax.legend()
plt.xlabel('excited sites')
plt.ylabel('probability')

def update_line(ax, tau):
    tau_states = meta_df[meta_df['tau'] == tau]
    state_prob = []
    for i in range(N):
        state_probability = sum(tau_states.loc[tau_states['excited_states'] == i]['probability'])
        state_prob.append(state_probability)
    
    line.set_ydata(state_prob)
    line.set_label(fr'$\tau=$ {tau:.3f}')
    ax.legend()

In [ ]:
from animator import ParticleAnimator
from matplotlib.animation import PillowWriter # requires pillowwriter package
filename = f'{N}-chain brute-force, A={A}, J1={J1}, J2={J2} t={min(tau_vals)}-{max(tau_vals)}.gif'

total_time = 5 # seconds
fps = len(tau_vals)/total_time
print(fps)

anim = ParticleAnimator(fig, PillowWriter, filename)
anim.set_display(fps, total_time)
frame_data = anim.generate_frame_data(tau_vals)
anim.animate_ax(ax, update_line, frame_data)
anim.make_animation()
anim.display()

Excellent! As we'd expect, higher temperatures increase the probability of mixed states occuring. Let's move on to simulation

In [ ]:
import random
def kinetic_monte_carlo(binary_lat, beta, total_time):
    t = 0.0
    time_series = [(t, binary_lat._states.copy())]
    
    while t < total_time:
        rates = []
        deltas = []
        indices = []
        
        # Compute rate for each possible flip
        for i in range(binary_lat._volume):
            delta_E = binary_lat.calculate_delta_energy(i)
            rate = np.exp(-beta * delta_E)
            rates.append(rate)
            deltas.append(delta_E)
            indices.append(i)
        
        R_total = sum(rates)
        if R_total == 0:
            break  # No moves possible

        # Sample time increment
        dt = -np.log(random.random()) / R_total
        t += dt

        # Choose site to flip
        r = random.random() * R_total
        acc = 0.0
        for i, rate in enumerate(rates):
            acc += rate
            if acc >= r:
                binary_lat._states[indices[i]] = 1 - binary_lat._states[indices[i]]
                break
        
        time_series.append((t, binary_lat._states.copy()))
    
    return time_series

dim = 1
N = 16
chain = lattice(dim, [N])

A, J1, J2 = 0.01, 4, 2
chain.make_cold_start(A=0.01, J1=4, J2=2, up=True)
#chain.make_hot_start(A=0.0, J1=3, J2=2)

tau = 3.5
beta = 1/tau

dynamics = np.array([b for a,b in kinetic_monte_carlo(binary_lat=chain, beta=beta, total_time=300)])

In [ ]:
plt.figure(figsize=(10,30))
sns.heatmap(np.array(dynamics), xticklabels=False, yticklabels=False, cbar=False)